# E3.1 — ROV vs NPV Fitness Function Comparison

Compares LSMC-based real options valuation (ROV) with static NPV as the fitness function, in terms of estimated portfolio value, computational cost and selected portfolios.

**Experiment:** 2 valuation modes × N seeds, using best encoding (E1.1) and best CHT (E2.1).

| Config | Valuation |
|--------|----------|
| rov | LSMC-based ROV with deferral option + go/no-go floor |
| npv | Static NPV at start_year, no deferral, no floor |

**Metrics:** HVI, IGD, ROV premium, relative computational cost, portfolio composition differences.

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import wilcoxon

def paired_wilcoxon(metric_a, seeds_a, metric_b, seeds_b):
    """Wilcoxon signed-rank paired by seed. Returns (p_value, n_common)."""
    seed_to_a = dict(zip(seeds_a, metric_a))
    seed_to_b = dict(zip(seeds_b, metric_b))
    common = sorted(set(seed_to_a) & set(seed_to_b))
    va = np.array([seed_to_a[s] for s in common])
    vb = np.array([seed_to_b[s] for s in common])
    if np.allclose(va, vb):
        return 1.0, len(common)
    return wilcoxon(va, vb, alternative="two-sided").pvalue, len(common)


def paired_diff(metric_a, seeds_a, metric_b, seeds_b):
    """Per-seed (A - B) differences, aligned by seed."""
    sa_to_v = dict(zip(seeds_a, metric_a))
    sb_to_v = dict(zip(seeds_b, metric_b))
    common = sorted(set(sa_to_v) & set(sb_to_v))
    return np.array([sa_to_v[s] - sb_to_v[s] for s in common])

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import (
    hypervolume_2d,
    inverted_generational_distance,
    construct_reference_front,
)

config_order = ["rov", "npv"]
config_labels = ["ROV (LSMC)", "Static NPV"]
colors = ["steelblue", "coral"]

## 1. Load Results

In [ ]:
runs = load_experiment_results("E3.1")
print(f"Loaded {len(runs)} runs")

configs = defaultdict(list)
for run in runs:
    s = run["summary"]
    if s["status"] != "completed":
        print(f"  SKIPPING failed run: {s['run_id']}")
        continue
    configs[s["config_name"]].append(run)

for name, run_list in sorted(configs.items()):
    times = [r["summary"]["elapsed_seconds"] for r in run_list]
    print(f"  {name:20s}: {len(run_list)} runs, avg time {np.mean(times):.1f}s")

## 2. HVI and IGD

In [ ]:
fronts_by_config = {}
for name, run_list in configs.items():
    fronts = []
    for run in run_list:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        if pf:
            obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
            fronts.append(obj)
        else:
            fronts.append(np.empty((0, 2)))
    fronts_by_config[name] = fronts

all_objectives = [f for fronts in fronts_by_config.values() for f in fronts if len(f) > 0]
reference_front = construct_reference_front(all_objectives)
all_combined = np.vstack(all_objectives)
ref_point = np.array([
    all_combined[:, 0].min() - abs(all_combined[:, 0].min()) * 0.1,
    all_combined[:, 1].min() - abs(all_combined[:, 1].min()) * 0.1,
])

seeds_by_config = {name: [r["summary"]["seed"] for r in run_list] for name, run_list in configs.items()}
hvi_by_config = {}
igd_by_config = {}
for name, fronts in fronts_by_config.items():
    hvi_by_config[name] = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    igd_by_config[name] = np.array([inverted_generational_distance(f, reference_front) for f in fronts])

print(f"{'Config':20s} {'HVI mean':>12s} {'HVI std':>10s} {'IGD mean':>12s} {'IGD std':>10s}")
print("-" * 66)
for name in config_order:
    if name not in hvi_by_config: continue
    hvi, igd = hvi_by_config[name], igd_by_config[name]
    print(f"{name:20s} {hvi.mean():12.1f} {hvi.std():10.1f} {igd.mean():12.4f} {igd.std():10.4f}")

## 3. Box Plots and Statistical Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
active = [c for c in config_order if c in hvi_by_config]
labels = [config_labels[config_order.index(c)] for c in active]

for ax, data_dict, ylabel, title in [
    (axes[0], hvi_by_config, "HVI", "Hypervolume Indicator (higher = better)"),
    (axes[1], igd_by_config, "IGD", "Inverted Generational Distance (lower = better)"),
]:
    data = [data_dict[c] for c in active]
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    for p, color in zip(bp["boxes"], colors):
        p.set_facecolor(color); p.set_alpha(0.7)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Significance test
if len(active) == 2:
    c1, c2 = active
    p_hvi, _ = paired_wilcoxon(hvi_by_config[c1], seeds_by_config[c1], hvi_by_config[c2], seeds_by_config[c2])
    p_igd, _ = paired_wilcoxon(igd_by_config[c1], seeds_by_config[c1], igd_by_config[c2], seeds_by_config[c2])
    print(f"\n{config_labels[0]} vs {config_labels[1]}:")
    print(f"  HVI: p = {p_hvi:.6f}")
    print(f"  IGD: p = {p_igd:.6f}")

## 4. Overlaid Pareto Fronts

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for i, name in enumerate(config_order):
    if name not in hvi_by_config: continue
    hvi_vals = hvi_by_config[name]
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))
    front = fronts_by_config[name][median_idx]
    if len(front) == 0: continue
    label = f"{config_labels[i]} (n = {len(front)})"
    ax.scatter(front[:, 1], front[:, 0], color=colors[i], label=label,
               s=70, alpha=0.75, edgecolor="black", linewidth=0.5)

ax.set_xlabel("CVaR (MNOK)"); ax.set_ylabel("Expected Value (MNOK)")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

## 5. Computational Cost Comparison

Relative wall-clock time per run: ROV (with LSMC) vs NPV (no LSMC).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
active = [c for c in config_order if c in configs]
labels = [config_labels[config_order.index(c)] for c in active]
time_data = [[r["summary"]["elapsed_seconds"] for r in configs[c]] for c in active]

bp = ax.boxplot(time_data, labels=labels, patch_artist=True)
for p, color in zip(bp["boxes"], colors):
    p.set_facecolor(color); p.set_alpha(0.7)
ax.set_ylabel("Wall-clock time (seconds)")
ax.set_title("Computational Cost Per Run")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Ratio
if len(active) == 2:
    t_rov = np.mean([r["summary"]["elapsed_seconds"] for r in configs[active[0]]])
    t_npv = np.mean([r["summary"]["elapsed_seconds"] for r in configs[active[1]]])
    print(f"Mean time — ROV: {t_rov:.1f}s, NPV: {t_npv:.1f}s")
    print(f"ROV/NPV cost ratio: {t_rov / t_npv:.2f}x")

## 6. Portfolio Composition Comparison

Which fields are selected under ROV vs NPV? Field selection frequency across all Pareto solutions.

In [ ]:
from src.optimizer.encoding import decode_particle

def get_field_selection_freq(run_list, config):
    """Fraction of Pareto solutions that develop each field, averaged across seeds."""
    n_fields = config["n_fields"]
    freq_per_seed = []
    for run in run_list:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        cfg = result["config"]
        encoding = cfg["pso"].get("encoding", None)
        if not pf:
            freq_per_seed.append(np.zeros(n_fields))
            continue
        developed = np.zeros((len(pf), n_fields))
        for j, entry in enumerate(pf):
            alloc = decode_particle(
                entry["position"], cfg["n_fields"], cfg["n_hosts"],
                cfg["max_start_year"], cfg["data_generation"]["max_wells"],
                encoding=encoding,
            )
            developed[j] = (alloc.assignment >= 0).astype(float)
        freq_per_seed.append(developed.mean(axis=0))
    return np.mean(freq_per_seed, axis=0)

active = [c for c in config_order if c in configs]
if active:
    # Get config from first run to know n_fields
    sample_result = load_run_result(configs[active[0]][0]["pickle_path"])
    cfg = sample_result["config"]
    n_fields = cfg["n_fields"]
    field_names = [f"F{i+1}" for i in range(n_fields)]

    # Muted light blue for ROV (constant at 1.0); keep NPV prominent
    plot_colors = {"rov": "lightsteelblue", "npv": "coral"}
    plot_alpha = {"rov": 0.6, "npv": 0.85}

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(n_fields)
    width = 0.35

    for i, name in enumerate(active):
        freq = get_field_selection_freq(configs[name], cfg)
        offset = -width/2 + i * width
        total = freq.sum()
        label = f"{config_labels[config_order.index(name)]} (avg {total:.1f}/{n_fields} fields)"
        ax.bar(x + offset, freq, width, label=label,
               color=plot_colors[name], alpha=plot_alpha[name])

    ax.set_xlabel("Field")
    ax.set_ylabel("Selection Frequency")
    ax.set_xticks(x); ax.set_xticklabels(field_names)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

## 7. Convergence Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, name in enumerate(config_order):
    if name not in configs: continue
    all_rov, all_cvar = [], []
    for run in configs[name]:
        hp = run["run_dir"] / "history.pkl"
        if hp.exists():
            with open(hp, "rb") as f: h = pickle.load(f)
            all_rov.append(h["best_rov"]); all_cvar.append(h["best_cvar"])
    if not all_rov: continue
    max_len = max(len(r) for r in all_rov)
    rov_arr = np.full((len(all_rov), max_len), np.nan)
    cvar_arr = np.full((len(all_cvar), max_len), np.nan)
    for j, (r, c) in enumerate(zip(all_rov, all_cvar)):
        rov_arr[j, :len(r)] = r; cvar_arr[j, :len(c)] = c
    iters = np.arange(1, max_len + 1)
    axes[0].plot(iters, np.nanmean(rov_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)
    axes[1].plot(iters, np.nanmean(cvar_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)

axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Best E[Value] (MNOK)")
axes[0].set_title("Convergence — Best Expected Value"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Best CVaR (MNOK)")
axes[1].set_title("Convergence — Best CVaR"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Summary

Selects the valuation method carried forward to E4.1. If ROV produces meaningfully different portfolios, ROV is used despite its higher cost. If the fronts are similar, NPV is preferred because it is cheaper.

In [ ]:
active = [c for c in config_order if c in hvi_by_config]
print("Ranking by mean HVI:")
for rank, (name, val) in enumerate(sorted([(n, hvi_by_config[n].mean()) for n in active], key=lambda x: -x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:20s} HVI = {val:.1f}")
print("\nRanking by mean IGD:")
for rank, (name, val) in enumerate(sorted([(n, igd_by_config[n].mean()) for n in active], key=lambda x: x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:20s} IGD = {val:.4f}")